## Bar graphs of various algorithm results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re
from pathlib import Path

DATA_DIR = Path('final_results_copy')


In [ ]:
SEQ_PATTERN = re.compile(
    r'\s*\((water only|water|fat fraction|fat_fraction|dixon|Dixon|both channels?|both)\)\s*$',
    re.IGNORECASE
)

def base_name(name: str) -> str:
    return SEQ_PATTERN.sub('', name).strip()

def extract_seq(name):
    m = re.search(r'\((water only|water|fat fraction|dixon|both[^)]*?)\)', name, re.IGNORECASE)
    if m:
        s = m.group(1).strip().lower()
        return {'water': 'Water', 'water only': 'Water',
                'fat fraction': 'Fat fraction',
                'dixon': 'Dixon'}.get(s, s.capitalize())
    return 'Single'

RENAME = {'Hirriririir': 'Multimodal-multiethnic'}

def display_name(name: str) -> str:
    return RENAME.get(base_name(name), base_name(name))

ALG_ORDER = [
    'MuscleMap WB',
    'MuscleMap Thigh',
    'MM WB + MedSAM bbox',
    'MM WB + SLM-SAM2',
    'MuSeg',
    'Multimodal-multiethnic',
    'MedCLIP-SAMv2 Text+Boxes',
    'MM WB + MedSAM mask',
    'MedCLIP-SAMv2',
    'Dafne + MedSAM',
    'Dafne',
    'MedSegDiff',
]

FAMILY = {
    'MuscleMap WB':             'U-Net',
    'MuscleMap Thigh':          'U-Net',
    'MuSeg':                    'nnU-Net',
    'Multimodal-multiethnic':   'SegResNet (MONAI)',
    'MM WB + MedSAM bbox':      'SAM-based',
    'MM WB + MedSAM mask':      'SAM-based',
    'MM WB + SLM-SAM2':         'SAM-based',
    'MedCLIP-SAMv2':            'SAM-based',
    'MedCLIP-SAMv2 Text+Boxes': 'SAM-based',
    'Dafne + MedSAM':           'Federated DL',
    'Dafne':                    'Federated DL',
    'MedSegDiff':               'Diffusion',
}

FAMILY_COLORS = {
    'U-Net':             '#1f77b4',
    'nnU-Net':           '#aec7e8',
    'SegResNet (MONAI)': '#9467bd',
    'SAM-based':         '#ff7f0e',
    'Federated DL':      '#2ca02c',
    'Diffusion':         '#d62728',
}

DATASETS = {
    'MyoSegmenTUM': 'overall_means_myosegmentum.csv',
    'Pathological': 'overall_means_P_only.csv',
    'AIPS':         'overall_means_asian.csv',
    'Sheffield':    'overall_means_sheffield.csv',
    'Augmented':    'overall_means_augmented.csv',
}

SEQ_ORDER   = ['Water', 'Fat fraction', 'Dixon', 'Single']
SEQ_HATCHES = {'Water': '', 'Fat fraction': '///', 'Dixon': 'xxx', 'Single': ''}


In [ ]:
# 5 datasets as subplots in a 2×3 grid, one shared legend below

NCOLS = 3
NROWS = 2
ds_list = list(DATASETS.items())   # 5 entries

fig, axes = plt.subplots(NROWS, NCOLS, figsize=(30, 12),
                          sharey=True, constrained_layout=False)
fig.subplots_adjust(hspace=0.55, wspace=0.12, bottom=0.22)

all_seq_present = set()
all_families_present = set()

for idx, (ds_name, fname) in enumerate(ds_list):
    row, col = divmod(idx, NCOLS)
    ax = axes[row, col]

    df_raw = pd.read_csv(DATA_DIR / fname)
    df_raw['base']     = df_raw['algorithm'].apply(display_name)
    df_raw['sequence'] = df_raw['algorithm'].apply(extract_seq)

    df_bar = df_raw[df_raw['base'].isin(ALG_ORDER)].copy()
    if df_bar.empty:
        ax.set_visible(False)
        continue

    alg_mean      = df_bar.groupby('base')['dice'].mean()
    alg_order_bar = alg_mean.sort_values(ascending=False).index.tolist()
    seq_present   = [s for s in SEQ_ORDER if s in df_bar['sequence'].unique()]
    all_seq_present.update(seq_present)
    all_families_present.update(FAMILY.get(a) for a in alg_order_bar)
    n_seq  = len(seq_present)
    x      = np.arange(len(alg_order_bar))
    width  = 0.8 / n_seq

    for k, seq in enumerate(seq_present):
        subset = df_bar[df_bar['sequence'] == seq].set_index('base')['dice']
        vals   = [subset.get(a, np.nan) for a in alg_order_bar]
        colors = [FAMILY_COLORS[FAMILY.get(a, 'U-Net')] for a in alg_order_bar]
        offset = (k - n_seq / 2 + 0.5) * width
        ax.bar(x + offset, vals, width * 0.92,
               color=colors, hatch=SEQ_HATCHES[seq],
               edgecolor='white', linewidth=0.5, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(alg_order_bar, rotation=40, ha='right', fontsize=11)
    ax.set_ylim(0, 1.0)
    ax.set_yticks([0.25, 0.50, 0.75, 1.0])
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(ds_name, fontsize=15, fontweight='bold')
    if col == 0:
        ax.set_ylabel('Dice Score', fontsize=13)

# Hide the unused 6th subplot
axes[1, 2].set_visible(False)

# ── Single combined legend ────────────────────────────────────────────────────
legend_handles = []

# Family colour patches
for fam, col in FAMILY_COLORS.items():
    if fam in all_families_present:
        legend_handles.append(
            mpatches.Patch(facecolor=col, label=fam, alpha=0.85)
        )

# Sequence hatch patches (grey fill so hatch is visible)
for seq in SEQ_ORDER:
    if seq in all_seq_present:
        legend_handles.append(
            mpatches.Patch(facecolor='#aaaaaa', hatch=SEQ_HATCHES[seq],
                           label=f'Sequence: {seq}', alpha=0.85,
                           edgecolor='black', linewidth=0.5)
        )

fig.legend(
    handles=legend_handles,
    loc='lower center',
    bbox_to_anchor=(0.5, 0.0),
    ncol=len(legend_handles),
    fontsize=12,
    title='Algorithm family / MRI sequence',
    title_fontsize=12,
    framealpha=0.9,
)

slug = 'bar_all_datasets_together'
plt.savefig(f'{slug}.pdf', bbox_inches='tight')
plt.savefig(f'{slug}.png', dpi=150, bbox_inches='tight')
plt.savefig(f'{slug}.tif', dpi=300, bbox_inches='tight',
            format='tiff', pil_kwargs={'compression': 'tiff_lzw'})
plt.show()
print(f'Saved {slug}  .pdf / .png / .tif')
